# 07_openalex — Uji cakupan dengan OpenAlex
Notebook ini hanya Python (tanpa R). Jalankan Sel 1, 2, lalu 3.

Tujuan: memastikan kelangkaan irisan L3 (multi-label × imbalance) dan L5 (imbalance × shift) bukan artefak cakupan Scopus. OpenAlex juga mencakup preprint (arXiv).

## Sel 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, requests, pandas as pd, re, time
os.chdir('/content/drive/MyDrive/bibliometrik-cxr')
print(os.getcwd())

## Sel 2 — Hitung S1 dan L1–L5 di OpenAlex, bandingkan dengan Scopus

In [ ]:
EMAIL = ""   # opsional: isi email Anda agar masuk "polite pool" OpenAlex (lebih stabil)
BASE = "https://api.openalex.org/works"

SSL   = '("semi-supervised" OR "semi supervised" OR "pseudo-label" OR "pseudo-labeling" OR "pseudo-labelling" OR "pseudo label" OR "consistency regularization" OR fixmatch OR mixmatch OR "mean teacher" OR "self-training")'
CXR   = '("chest x-ray" OR "chest xray" OR "chest radiograph" OR "chest radiography" OR cxr OR chexpert OR "chestx-ray14" OR "mimic-cxr" OR "thoracic disease")'
ML    = '("multi-label" OR multilabel OR "multi label")'
IMB   = '(imbalance OR imbalanced OR "long-tailed" OR "long-tail" OR "long tail" OR "rare disease" OR "minority class")'
SHIFT = '("continual learning" OR lifelong OR "incremental learning" OR "domain shift" OR "distribution shift" OR "label shift" OR "prior shift")'

SETS = {
  "S1": f"{SSL} AND {CXR}",
  "L1": f"{SSL} AND {CXR} AND {ML}",
  "L2": f"{SSL} AND {CXR} AND {IMB}",
  "L3": f"{SSL} AND {CXR} AND {ML} AND {IMB}",
  "L4": f"{SSL} AND {CXR} AND {SHIFT}",
  "L5": f"{SSL} AND {CXR} AND {IMB} AND {SHIFT}",
}
SCOPUS = {"S1": 207, "L1": 28, "L2": 23, "L3": 6, "L4": 10, "L5": 3}

def query(q, extra=None, per_page=1, cursor=None, group_by=None):
    flt = f"title_and_abstract.search:{q},from_publication_date:2017-01-01"
    p = {"filter": flt, "per-page": per_page}
    if EMAIL: p["mailto"] = EMAIL
    if cursor: p["cursor"] = cursor
    if group_by: p["group_by"] = group_by
    r = requests.get(BASE, params=p, timeout=60)
    r.raise_for_status()
    return r.json()

rows = []
for k, q in SETS.items():
    js = query(q, group_by="type")
    total = sum(g["count"] for g in js["group_by"])
    per_type = {g["key_display_name"]: g["count"] for g in js["group_by"]}
    rows.append({"set": k, "openalex_total": total,
                 "openalex_preprint": per_type.get("preprint", 0),
                 "openalex_non_preprint": total - per_type.get("preprint", 0),
                 "scopus_sebelum_screening": SCOPUS[k]})
    time.sleep(0.3)
counts = pd.DataFrame(rows)
counts["L_pct_dari_S1_openalex"] = (100 * counts.openalex_total / counts.openalex_total.iloc[0]).round(1)
print(counts.to_string(index=False))

## Sel 3 — Daftar dokumen L3 dan L5, cek tumpang tindih dengan Scopus

In [ ]:
def fetch_all(q):
    out, cursor = [], "*"
    while cursor:
        js = query(q, per_page=200, cursor=cursor)
        for w in js["results"]:
            out.append({"judul": w.get("display_name"), "tahun": w.get("publication_year"),
                        "tipe": w.get("type"), "doi": (w.get("doi") or "").replace("https://doi.org/", "").lower(),
                        "venue": ((w.get("primary_location") or {}).get("source") or {}).get("display_name"),
                        "openalex_id": w.get("id")})
        cursor = js["meta"].get("next_cursor")
        if not js["results"]: break
        time.sleep(0.3)
    return pd.DataFrame(out)

sc = pd.read_csv("02_raw/scopus_S1.csv", encoding="utf-8-sig")
scopus_doi = set(sc["DOI"].dropna().str.lower())
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
scopus_judul = set(sc["Title"].map(norm))

hasil = {}
for k in ["L3", "L5"]:
    df = fetch_all(SETS[k])
    if len(df):
        df["ada_di_scopus_S1"] = df.apply(lambda r: (r.doi in scopus_doi) or (norm(r.judul) in scopus_judul), axis=1)
    hasil[k] = df
    print(f"\n=== {k}: {len(df)} dokumen di OpenAlex; "
          f"{int(df['ada_di_scopus_S1'].sum()) if len(df) else 0} sudah ada di corpus Scopus ===")
    if len(df):
        print(df[["tahun", "tipe", "ada_di_scopus_S1", "judul"]].to_string(index=False, max_colwidth=90))

with pd.ExcelWriter("05_figures/openalex_coverage.xlsx") as w:
    counts.to_excel(w, sheet_name="jumlah", index=False)
    for k, df in hasil.items(): df.to_excel(w, sheet_name=k, index=False)
print("\nTersimpan: 05_figures/openalex_coverage.xlsx")